In [5]:
# ==========================================
# 0. UNIVERSAL DEPENDENCY INSTALLATION
# ==========================================
import subprocess
import sys

def install_dependencies():
    packages = [
        "torch", "transformers", "accelerate", "sentencepiece", "protobuf",
        "scikit-learn", "scipy", "pandas", "numpy"
    ]
    for pkg in packages:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install_dependencies()

# ==========================================
# 1. SETUP & SAMPLING
# ==========================================
import pandas as pd
import numpy as np
import torch
import re
from transformers import pipeline
from sklearn.metrics import accuracy_score
import gc

print("Loading Level 1 Core Political data...")
df = pd.read_csv('/kaggle/input/notebooks/shivanguniyal/merging-cleaned-labels-with-bertopic-output/FINAL_Hindi_Level1_Clean.csv')
text_col = 'clean_text' if 'clean_text' in df.columns else 'article_text'

# Sample 1000 articles for the validation set
df_sample = df.sample(1000, random_state=42).reset_index(drop=True)
print(f"✓ Sampled {len(df_sample)} articles for Threshold Optimization.")

device = 0 if torch.cuda.is_available() else -1
pipe = pipeline("zero-shot-classification", 
                model="MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli", 
                device=device, batch_size=16)

candidate_labels = ["critical", "neutral", "supportive"]
hypothesis_template = "The overall evaluative orientation conveyed by this news article is {}."
label_to_idx = {label: idx for idx, label in enumerate(candidate_labels)}

# ==========================================
# 2. RUN TRUNCATED PASS (Get Probabilities)
# ==========================================
print("\n--- Running Truncated Pass ---")
texts_trunc = [str(t)[:1500] for t in df_sample[text_col].tolist()]

trunc_probs = np.zeros((len(df_sample), 3))
super_chunk_size = 1000
for i in range(0, len(texts_trunc), super_chunk_size):
    batch = texts_trunc[i:i+super_chunk_size]
    res = pipe(batch, candidate_labels, hypothesis_template=hypothesis_template, multi_label=False, truncation=True)
    for j, r in enumerate(res):
        for label, score in zip(r['labels'], r['scores']):
            trunc_probs[i+j, label_to_idx[label]] = score

df_sample['trunc_label'] = [candidate_labels[idx] for idx in np.argmax(trunc_probs, axis=1)]
df_sample['trunc_conf'] = np.max(trunc_probs, axis=1)

# ==========================================
# 3. RUN HIERARCHICAL PASS (The "Gold Standard")
# ==========================================
print("\n--- Running Hierarchical Pass (Gold Standard) ---")
def get_hierarchical_chunks(text, chunk_size=1000):
    text = str(text)
    sentences = re.split(r'(?<=[.!?।])\s+', text)
    chunks, current_chunk = [], ""
    for sent in sentences:
        if len(current_chunk) + len(sent) > chunk_size:
            if current_chunk: chunks.append(current_chunk)
            current_chunk = sent
        else:
            current_chunk += " " + sent if current_chunk else sent
    if current_chunk: chunks.append(current_chunk)
    return chunks if chunks else [text]

df_sample['chunks'] = df_sample[text_col].apply(get_hierarchical_chunks)

all_chunks = []
chunk_mapping = [] 
for i, chunks in enumerate(df_sample['chunks']):
    for chunk in chunks:
        all_chunks.append(chunk)
        chunk_mapping.append(i)

hier_sums = {idx: np.zeros(3) for idx in range(len(df_sample))}
hier_counts = {idx: 0 for idx in range(len(df_sample))}

for i in range(0, len(all_chunks), super_chunk_size):
    batch = all_chunks[i:i+super_chunk_size]
    res = pipe(batch, candidate_labels, hypothesis_template=hypothesis_template, multi_label=False, truncation=True)
    for j, r in enumerate(res):
        orig_idx = chunk_mapping[i+j]
        for label, score in zip(r['labels'], r['scores']):
            hier_sums[orig_idx][label_to_idx[label]] += score
        hier_counts[orig_idx] += 1

hier_probs = np.zeros((len(df_sample), 3))
for idx in range(len(df_sample)):
    if hier_counts[idx] > 0:
        hier_probs[idx] = hier_sums[idx] / hier_counts[idx]

df_sample['hier_label'] = [candidate_labels[idx] for idx in np.argmax(hier_probs, axis=1)]

# ==========================================
# 4. EMPIRICAL THRESHOLD OPTIMIZATION
# ==========================================
print("\n" + "="*80)
print("📊 EMPIRICAL THRESHOLD TRADE-OFF ANALYSIS")
print("="*80)
print(f"{'Threshold':<10} | {'% Escalated':<12} | {'% Accepted':<12} | {'Accepted Acc (vs Hier)':<22} | {'Overall Pipeline Acc':<20}")
print("-" * 80)

results = []
# Test thresholds from 0.50 to 0.95
for t in np.arange(0.50, 0.96, 0.05):
    accepted_mask = df_sample['trunc_conf'] >= t
    
    n_total = len(df_sample)
    n_escalated = (~accepted_mask).sum()
    n_accepted = accepted_mask.sum()
    
    if n_accepted > 0:
        # How accurate is the Truncated model when we TRUST it?
        acc_accepted = accuracy_score(df_sample.loc[accepted_mask, 'hier_label'], df_sample.loc[accepted_mask, 'trunc_label'])
    else:
        acc_accepted = 0.0
        
    # Overall Pipeline Accuracy: 
    # (Accepted articles that match Hierarchical + Escalated articles that match Hierarchical) / Total
    # Note: Escalated articles match Hierarchical 100% of the time by definition.
    overall_acc = ((n_accepted * acc_accepted) + (n_escalated * 1.0)) / n_total
    
    results.append({
        'Threshold': round(t, 2),
        'Escalation_Rate': n_escalated / n_total,
        'Acceptance_Rate': n_accepted / n_total,
        'Accepted_Accuracy': acc_accepted,
        'Overall_Accuracy': overall_acc
    })
    
    print(f"{round(t, 2):<10} | {n_escalated / n_total:<12.1%} | {n_accepted / n_total:<12.1%} | {acc_accepted:<22.1%} | {overall_acc:<20.1%}")

print("="*80)
print("\n💡 HOW TO READ THIS TABLE:")
print("Look for the 'Elbow': The threshold where 'Accepted Accuracy' is high (e.g., >90%),")
print("but the 'Escalation Rate' hasn't exploded yet. That is your empirically justified threshold.")

del pipe
gc.collect()
torch.cuda.empty_cache()

Loading Level 1 Core Political data...
✓ Sampled 1000 articles for Threshold Optimization.


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Running Truncated Pass ---


KeyboardInterrupt: 

In [3]:
# ==========================================
# 0. UNIVERSAL DEPENDENCY INSTALLATION
# ==========================================
import subprocess
import sys

def install_dependencies():
    packages = [
        "torch", "transformers", "accelerate", "sentencepiece", "protobuf",
        "scikit-learn", "scipy", "pandas", "numpy"
    ]
    for pkg in packages:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install_dependencies()

# ==========================================
# 1. SETUP & SAMPLING
# ==========================================
import pandas as pd
import numpy as np
import torch
import re
from transformers import pipeline
from sklearn.metrics import accuracy_score
import gc

print("Loading Level 1 Core Political data...")
df = pd.read_csv('/kaggle/input/notebooks/shivanguniyal/merging-cleaned-labels-with-bertopic-output/FINAL_English_Level1_Clean.csv')
text_col = 'clean_text' if 'clean_text' in df.columns else 'article_text'

# Sample 1000 articles for the validation set
df_sample = df.sample(1000, random_state=42).reset_index(drop=True)
print(f"✓ Sampled {len(df_sample)} articles for Dual-Elbow Optimization.")

device = 0 if torch.cuda.is_available() else -1
pipe = pipeline("zero-shot-classification", 
                model="MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli", 
                device=device, batch_size=16)

candidate_labels = ["critical", "neutral", "supportive"]
hypothesis_template = "The overall evaluative orientation conveyed by this news article is {}."
label_to_idx = {label: idx for idx, label in enumerate(candidate_labels)}

# ==========================================
# 2. RUN TRUNCATED PASS (Get Probabilities)
# ==========================================
print("\n--- Running Truncated Pass ---")
texts_trunc = [str(t)[:1500] for t in df_sample[text_col].tolist()]

trunc_probs = np.zeros((len(df_sample), 3))
super_chunk_size = 1000
for i in range(0, len(texts_trunc), super_chunk_size):
    batch = texts_trunc[i:i+super_chunk_size]
    res = pipe(batch, candidate_labels, hypothesis_template=hypothesis_template, multi_label=False, truncation=True)
    for j, r in enumerate(res):
        for label, score in zip(r['labels'], r['scores']):
            trunc_probs[i+j, label_to_idx[label]] = score

df_sample['trunc_label'] = [candidate_labels[idx] for idx in np.argmax(trunc_probs, axis=1)]
df_sample['trunc_conf'] = np.max(trunc_probs, axis=1)

# Calculate Margin (Top Probability - Second Highest Probability)
sorted_trunc_probs = np.sort(trunc_probs, axis=1)
df_sample['trunc_margin'] = sorted_trunc_probs[:, -1] - sorted_trunc_probs[:, -2]

# ==========================================
# 3. RUN HIERARCHICAL PASS (The "Gold Standard")
# ==========================================
print("\n--- Running Hierarchical Pass (Gold Standard) ---")
def get_hierarchical_chunks(text, chunk_size=1000):
    text = str(text)
    sentences = re.split(r'(?<=[.!?।])\s+', text)
    chunks, current_chunk = [], ""
    for sent in sentences:
        if len(current_chunk) + len(sent) > chunk_size:
            if current_chunk: chunks.append(current_chunk)
            current_chunk = sent
        else:
            current_chunk += " " + sent if current_chunk else sent
    if current_chunk: chunks.append(current_chunk)
    return chunks if chunks else [text]

df_sample['chunks'] = df_sample[text_col].apply(get_hierarchical_chunks)

all_chunks = []
chunk_mapping = [] 
for i, chunks in enumerate(df_sample['chunks']):
    for chunk in chunks:
        all_chunks.append(chunk)
        chunk_mapping.append(i)

hier_sums = {idx: np.zeros(3) for idx in range(len(df_sample))}
hier_counts = {idx: 0 for idx in range(len(df_sample))}

for i in range(0, len(all_chunks), super_chunk_size):
    batch = all_chunks[i:i+super_chunk_size]
    res = pipe(batch, candidate_labels, hypothesis_template=hypothesis_template, multi_label=False, truncation=True)
    for j, r in enumerate(res):
        orig_idx = chunk_mapping[i+j]
        for label, score in zip(r['labels'], r['scores']):
            hier_sums[orig_idx][label_to_idx[label]] += score
        hier_counts[orig_idx] += 1

hier_probs = np.zeros((len(df_sample), 3))
for idx in range(len(df_sample)):
    if hier_counts[idx] > 0:
        hier_probs[idx] = hier_sums[idx] / hier_counts[idx]

df_sample['hier_label'] = [candidate_labels[idx] for idx in np.argmax(hier_probs, axis=1)]

# ==========================================
# 4. EMPIRICAL THRESHOLD OPTIMIZATION (DUAL ELBOW)
# ==========================================
print("\n" + "="*80)
print("📊 TABLE 1: CONFIDENCE THRESHOLD TRADE-OFF")
print("="*80)
print(f"{'Threshold':<10} | {'% Escalated':<12} | {'% Accepted':<12} | {'Accepted Acc (vs Hier)':<22} | {'Overall Pipeline Acc':<20}")
print("-" * 80)

for t in np.arange(0.50, 0.96, 0.05):
    accepted_mask = df_sample['trunc_conf'] >= t
    n_total = len(df_sample)
    n_escalated = (~accepted_mask).sum()
    n_accepted = accepted_mask.sum()
    
    acc_accepted = accuracy_score(df_sample.loc[accepted_mask, 'hier_label'], df_sample.loc[accepted_mask, 'trunc_label']) if n_accepted > 0 else 0.0
    overall_acc = ((n_accepted * acc_accepted) + (n_escalated * 1.0)) / n_total
    
    print(f"{round(t, 2):<10} | {n_escalated / n_total:<12.1%} | {n_accepted / n_total:<12.1%} | {acc_accepted:<22.1%} | {overall_acc:<20.1%}")


print("\n" + "="*80)
print("📊 TABLE 2: MARGIN THRESHOLD TRADE-OFF (The 'Coin-Flip' Catcher)")
print("="*80)
print(f"{'Margin':<10} | {'% Escalated':<12} | {'% Accepted':<12} | {'Accepted Acc (vs Hier)':<22} | {'Overall Pipeline Acc':<20}")
print("-" * 80)

for m in np.arange(0.05, 0.55, 0.05):
    # Accepted if Margin is GREATER THAN OR EQUAL TO the threshold
    accepted_mask = df_sample['trunc_margin'] >= m
    n_total = len(df_sample)
    n_escalated = (~accepted_mask).sum()
    n_accepted = accepted_mask.sum()
    
    acc_accepted = accuracy_score(df_sample.loc[accepted_mask, 'hier_label'], df_sample.loc[accepted_mask, 'trunc_label']) if n_accepted > 0 else 0.0
    overall_acc = ((n_accepted * acc_accepted) + (n_escalated * 1.0)) / n_total
    
    print(f"{round(m, 2):<10} | {n_escalated / n_total:<12.1%} | {n_accepted / n_total:<12.1%} | {acc_accepted:<22.1%} | {overall_acc:<20.1%}")

print("="*80)
print("\n💡 HOW TO READ THESE TABLES:")
print("1. CONFIDENCE: Look for the threshold where Accepted Accuracy crosses ~88-90% without escalating >60% of the corpus.")
print("2. MARGIN: Look for the threshold where Accepted Accuracy stabilizes. A margin of 0.15 or 0.20 usually catches the dangerous 50/50 splits.")
print("3. In production, you will escalate if (Conf < X) OR (Margin < Y).")

del pipe
gc.collect()
torch.cuda.empty_cache()

Loading Level 1 Core Political data...
✓ Sampled 1000 articles for Dual-Elbow Optimization.


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Running Truncated Pass ---

--- Running Hierarchical Pass (Gold Standard) ---

📊 TABLE 1: CONFIDENCE THRESHOLD TRADE-OFF
Threshold  | % Escalated  | % Accepted   | Accepted Acc (vs Hier) | Overall Pipeline Acc
--------------------------------------------------------------------------------
0.5        | 20.9%        | 79.1%        | 83.1%                  | 86.6%               
0.55       | 32.8%        | 67.2%        | 84.8%                  | 89.8%               
0.6        | 47.8%        | 52.2%        | 86.8%                  | 93.1%               
0.65       | 61.1%        | 38.9%        | 88.9%                  | 95.7%               
0.7        | 73.9%        | 26.1%        | 92.3%                  | 98.0%               
0.75       | 83.4%        | 16.6%        | 94.6%                  | 99.1%               
0.8        | 91.3%        | 8.7%         | 94.3%                  | 99.5%               
0.85       | 95.3%        | 4.7%         | 97.9%                  | 99.9%        

In [4]:
# ==========================================
# 0. UNIVERSAL DEPENDENCY INSTALLATION
# ==========================================
import subprocess
import sys

def install_dependencies():
    packages = [
        "torch", "transformers", "accelerate", "sentencepiece", "protobuf",
        "scikit-learn", "scipy", "pandas", "numpy"
    ]
    for pkg in packages:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install_dependencies()

# ==========================================
# 1. SETUP & SAMPLING
# ==========================================
import pandas as pd
import numpy as np
import torch
import re
from transformers import pipeline
from sklearn.metrics import accuracy_score
import gc

print("Loading Level 1 Core Political data...")
df = pd.read_csv('/kaggle/input/notebooks/shivanguniyal/merging-cleaned-labels-with-bertopic-output/FINAL_English_Level1_Clean.csv')
text_col = 'clean_text' if 'clean_text' in df.columns else 'article_text'

# Sample 1000 articles
df_sample = df.sample(1000, random_state=42).reset_index(drop=True)
print(f"✓ Sampled {len(df_sample)} articles for Combined Rule Analysis.")

device = 0 if torch.cuda.is_available() else -1
pipe = pipeline("zero-shot-classification", 
                model="MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli", 
                device=device, batch_size=16)

candidate_labels = ["critical", "neutral", "supportive"]
hypothesis_template = "The overall evaluative orientation conveyed by this news article is {}."
label_to_idx = {label: idx for idx, label in enumerate(candidate_labels)}

# ==========================================
# 2. RUN TRUNCATED PASS
# ==========================================
print("\n--- Running Truncated Pass ---")
texts_trunc = [str(t)[:1500] for t in df_sample[text_col].tolist()]

trunc_probs = np.zeros((len(df_sample), 3))
super_chunk_size = 1000
for i in range(0, len(texts_trunc), super_chunk_size):
    batch = texts_trunc[i:i+super_chunk_size]
    res = pipe(batch, candidate_labels, hypothesis_template=hypothesis_template, multi_label=False, truncation=True)
    for j, r in enumerate(res):
        for label, score in zip(r['labels'], r['scores']):
            trunc_probs[i+j, label_to_idx[label]] = score

df_sample['trunc_label'] = [candidate_labels[idx] for idx in np.argmax(trunc_probs, axis=1)]
df_sample['trunc_conf'] = np.max(trunc_probs, axis=1)

sorted_trunc_probs = np.sort(trunc_probs, axis=1)
df_sample['trunc_margin'] = sorted_trunc_probs[:, -1] - sorted_trunc_probs[:, -2]

# ==========================================
# 3. RUN HIERARCHICAL PASS (Gold Standard)
# ==========================================
print("\n--- Running Hierarchical Pass (Gold Standard) ---")
def get_hierarchical_chunks(text, chunk_size=1000):
    text = str(text)
    sentences = re.split(r'(?<=[.!?।])\s+', text)
    chunks, current_chunk = [], ""
    for sent in sentences:
        if len(current_chunk) + len(sent) > chunk_size:
            if current_chunk: chunks.append(current_chunk)
            current_chunk = sent
        else:
            current_chunk += " " + sent if current_chunk else sent
    if current_chunk: chunks.append(current_chunk)
    return chunks if chunks else [text]

df_sample['chunks'] = df_sample[text_col].apply(get_hierarchical_chunks)

all_chunks = []
chunk_mapping = [] 
for i, chunks in enumerate(df_sample['chunks']):
    for chunk in chunks:
        all_chunks.append(chunk)
        chunk_mapping.append(i)

hier_sums = {idx: np.zeros(3) for idx in range(len(df_sample))}
hier_counts = {idx: 0 for idx in range(len(df_sample))}

for i in range(0, len(all_chunks), super_chunk_size):
    batch = all_chunks[i:i+super_chunk_size]
    res = pipe(batch, candidate_labels, hypothesis_template=hypothesis_template, multi_label=False, truncation=True)
    for j, r in enumerate(res):
        orig_idx = chunk_mapping[i+j]
        for label, score in zip(r['labels'], r['scores']):
            hier_sums[orig_idx][label_to_idx[label]] += score
        hier_counts[orig_idx] += 1

hier_probs = np.zeros((len(df_sample), 3))
for idx in range(len(df_sample)):
    if hier_counts[idx] > 0:
        hier_probs[idx] = hier_sums[idx] / hier_counts[idx]

df_sample['hier_label'] = [candidate_labels[idx] for idx in np.argmax(hier_probs, axis=1)]

# ==========================================
# 4. COMBINED RULE ANALYSIS
# ==========================================
print("\n" + "="*80)
print("📊 COMBINED ESCALATION RULE ANALYSIS")
print("="*80)

n_total = len(df_sample)

def evaluate_rule(mask, rule_name):
    n_escalated = mask.sum()
    n_accepted = (~mask).sum()
    
    if n_accepted > 0:
        # Accuracy of the accepted articles against the Hierarchical Gold Standard
        acc_accepted = accuracy_score(df_sample.loc[~mask, 'hier_label'], df_sample.loc[~mask, 'trunc_label'])
    else:
        acc_accepted = 0.0
        
    overall_acc = ((n_accepted * acc_accepted) + (n_escalated * 1.0)) / n_total
    
    print(f"\n🔹 {rule_name}")
    print(f"   % Escalated:        {n_escalated / n_total:.1%} ({n_escalated:,} articles)")
    print(f"   % Accepted:         {n_accepted / n_total:.1%} ({n_accepted:,} articles)")
    print(f"   Accepted Accuracy:  {acc_accepted:.1%}")
    print(f"   Overall Pipeline:   {overall_acc:.1%}")
    return n_escalated, acc_accepted, overall_acc

# Rule A: Standalone Confidence
mask_conf = df_sample['trunc_conf'] < 0.65
evaluate_rule(mask_conf, "Rule A: Standalone Confidence (< 0.65)")

# Rule B: Standalone Margin
mask_margin = df_sample['trunc_margin'] < 0.15
evaluate_rule(mask_margin, "Rule B: Standalone Margin (< 0.15)")

# Rule C: Combined (The Reviewer's Recommendation)
mask_combined = (df_sample['trunc_conf'] < 0.65) | (df_sample['trunc_margin'] < 0.15)
evaluate_rule(mask_combined, "Rule C: COMBINED (Conf < 0.65 OR Margin < 0.15)")

print("\n" + "="*80)
print("💡 CONCLUSION:")
print("If Rule C increases Accepted Accuracy significantly while only adding a small")
print("percentage to the Escalation Rate compared to Rule A, it is the mathematically")
print("dominant and most defensible choice for your methodology.")
print("="*80)

del pipe
gc.collect()
torch.cuda.empty_cache()

Loading Level 1 Core Political data...
✓ Sampled 1000 articles for Combined Rule Analysis.


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Running Truncated Pass ---

--- Running Hierarchical Pass (Gold Standard) ---

📊 COMBINED ESCALATION RULE ANALYSIS

🔹 Rule A: Standalone Confidence (< 0.65)
   % Escalated:        61.1% (611 articles)
   % Accepted:         38.9% (389 articles)
   Accepted Accuracy:  88.9%
   Overall Pipeline:   95.7%

🔹 Rule B: Standalone Margin (< 0.15)
   % Escalated:        22.6% (226 articles)
   % Accepted:         77.4% (774 articles)
   Accepted Accuracy:  83.6%
   Overall Pipeline:   87.3%

🔹 Rule C: COMBINED (Conf < 0.65 OR Margin < 0.15)
   % Escalated:        61.1% (611 articles)
   % Accepted:         38.9% (389 articles)
   Accepted Accuracy:  88.9%
   Overall Pipeline:   95.7%

💡 CONCLUSION:
If Rule C increases Accepted Accuracy significantly while only adding a small
percentage to the Escalation Rate compared to Rule A, it is the mathematically
dominant and most defensible choice for your methodology.


In [5]:
# ==========================================
# 0. UNIVERSAL DEPENDENCY INSTALLATION
# ==========================================
import subprocess
import sys

def install_dependencies():
    packages = ["torch", "transformers", "accelerate", "sentencepiece", "protobuf", "scikit-learn", "pandas", "numpy"]
    for pkg in packages:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install_dependencies()

# ==========================================
# 1. LOAD HUMAN-CODED DATA
# ==========================================
import pandas as pd
import numpy as np
import torch
import re
from transformers import pipeline
from sklearn.metrics import accuracy_score

# 👇 POINT THIS TO YOUR COMPLETED HUMAN-CODED CSV 👇
HUMAN_CSV_PATH = '/kaggle/input/datasets/shivanguniyal/prong3/prong3_blinded_audit.csv' 

df_human = pd.read_csv(HUMAN_CSV_PATH)

# Dynamically find the text and human label columns
text_col = next((c for c in df_human.columns if 'text' in c.lower() or 'clean' in c.lower()), None)
human_col = next((c for c in df_human.columns if 'human' in c.lower() or 'reference' in c.lower() or 'coder' in c.lower()), None)

# Filter to only rows where you actually entered a valid label
valid_labels = ['critical', 'neutral', 'supportive']
df_human[human_col] = df_human[human_col].str.lower().str.strip()
df_eval = df_human[df_human[human_col].isin(valid_labels)].reset_index(drop=True)

print(f"✓ Loaded {len(df_eval)} human-coded articles for calibration analysis.")

# ==========================================
# 2. RAPID RE-SCORING (To get exact probabilities)
# ==========================================
device = 0 if torch.cuda.is_available() else -1
pipe = pipeline("zero-shot-classification", 
                model="MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli", 
                device=device, batch_size=16)

candidate_labels = ["critical", "neutral", "supportive"]
hypothesis_template = "The overall evaluative orientation conveyed by this news article is {}."
label_to_idx = {label: idx for idx, label in enumerate(candidate_labels)}

def get_chunks(text, chunk_size=1000):
    text = str(text)
    sentences = re.split(r'(?<=[.!?।])\s+', text)
    chunks, current_chunk = [], ""
    for sent in sentences:
        if len(current_chunk) + len(sent) > chunk_size:
            if current_chunk: chunks.append(current_chunk)
            current_chunk = sent
        else:
            current_chunk += " " + sent if current_chunk else sent
    if current_chunk: chunks.append(current_chunk)
    return chunks if chunks else [text]

# A. Truncated Pass
texts_trunc = [str(t)[:1500] for t in df_eval[text_col].tolist()]
res_trunc = pipe(texts_trunc, candidate_labels, hypothesis_template=hypothesis_template, multi_label=False, truncation=True)

trunc_probs = np.zeros((len(df_eval), 3))
for i, r in enumerate(res_trunc):
    for label, score in zip(r['labels'], r['scores']):
        trunc_probs[i, label_to_idx[label]] = score

df_eval['trunc_label'] = [candidate_labels[idx] for idx in np.argmax(trunc_probs, axis=1)]
df_eval['trunc_conf'] = np.max(trunc_probs, axis=1)

# B. Hierarchical Pass
all_chunks = []
chunk_mapping = []
for i, text in enumerate(df_eval[text_col]):
    for chunk in get_chunks(text):
        all_chunks.append(chunk)
        chunk_mapping.append(i)

res_hier = pipe(all_chunks, candidate_labels, hypothesis_template=hypothesis_template, multi_label=False, truncation=True)

hier_sums = {idx: np.zeros(3) for idx in range(len(df_eval))}
hier_counts = {idx: 0 for idx in range(len(df_eval))}
for j, r in enumerate(res_hier):
    orig_idx = chunk_mapping[j]
    for label, score in zip(r['labels'], r['scores']):
        hier_sums[orig_idx][label_to_idx[label]] += score
    hier_counts[orig_idx] += 1

hier_probs = np.zeros((len(df_eval), 3))
for idx in range(len(df_eval)):
    if hier_counts[idx] > 0:
        hier_probs[idx] = hier_sums[idx] / hier_counts[idx]

df_eval['hier_label'] = [candidate_labels[idx] for idx in np.argmax(hier_probs, axis=1)]
df_eval['hier_conf'] = np.max(hier_probs, axis=1)

# ==========================================
# 3. APPLY CASCADE RULE & EVALUATE
# ==========================================
# Cascade Logic: If Truncated Conf >= 0.65, use Truncated. Else, use Hierarchical.
df_eval['final_machine_label'] = df_eval['trunc_label']
df_eval['final_conf'] = df_eval['trunc_conf']
df_eval['method_used'] = 'Truncated'

mask_escalated = df_eval['trunc_conf'] < 0.65
df_eval.loc[mask_escalated, 'final_machine_label'] = df_eval.loc[mask_escalated, 'hier_label']
df_eval.loc[mask_escalated, 'final_conf'] = df_eval.loc[mask_escalated, 'hier_conf']
df_eval.loc[mask_escalated, 'method_used'] = 'Hierarchical'

# ==========================================
# 4. OUTPUT 1: THE CALIBRATION CURVE
# ==========================================
print("\n" + "="*70)
print("📊 TABLE 1: CONFIDENCE CALIBRATION VS HUMAN AGREEMENT")
print("="*70)
print("If the model is well-calibrated, Human Agreement should rise with Confidence.")
print("-"*70)

bins = [0, 0.50, 0.65, 0.80, 0.95, 1.01]
labels_bin = ['<0.50 (Guessing)', '0.50-0.65 (Uncertain)', '0.65-0.80 (Moderate)', '0.80-0.95 (High)', '0.95-1.00 (Very High)']
df_eval['conf_bin'] = pd.cut(df_eval['final_conf'], bins=bins, labels=labels_bin, right=False)

print(f"{'Confidence Bin':<25} | {'N Articles':<10} | {'Human Agreement':<15} | {'Method Used Mostly'}")
print("-"*70)

for b in labels_bin:
    subset = df_eval[df_eval['conf_bin'] == b]
    if len(subset) > 0:
        acc = accuracy_score(subset[human_col], subset['final_machine_label'])
        method_split = subset['method_used'].value_counts().to_dict()
        print(f"{b:<25} | {len(subset):<10} | {acc:<15.1%} | {method_split}")

# ==========================================
# 5. OUTPUT 2: HIGH-CONFIDENCE FAILURE AUTOPSY
# ==========================================
print("\n" + "="*70)
print("🔍 TABLE 2: HIGH-CONFIDENCE FAILURES (The 'Why did it fail?' Autopsy)")
print("="*70)
print("Articles where the model was >75% confident, but the Human disagreed.")
print("-"*70)

failures = df_eval[(df_eval['final_machine_label'] != df_eval[human_col]) & (df_eval['final_conf'] > 0.75)]

if len(failures) == 0:
    print("🎉 ZERO high-confidence failures! The model's certainty perfectly aligns with human judgment.")
else:
    print(f"Found {len(failures)} high-confidence disagreements:\n")
    for idx, row in failures.iterrows():
        print(f"❌ Machine: {row['final_machine_label'].upper()} ({row['final_conf']:.2f} via {row['method_used']}) | 👤 Human: {row[human_col].upper()}")
        print(f"📰 Text Snippet: {str(row[text_col])[:200].replace(chr(10), ' ')}...")
        print("-"*70)

del pipe
torch.cuda.empty_cache()

✓ Loaded 99 human-coded articles for calibration analysis.


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



📊 TABLE 1: CONFIDENCE CALIBRATION VS HUMAN AGREEMENT
If the model is well-calibrated, Human Agreement should rise with Confidence.
----------------------------------------------------------------------
Confidence Bin            | N Articles | Human Agreement | Method Used Mostly
----------------------------------------------------------------------
<0.50 (Guessing)          | 63         | 46.0%           | {'Hierarchical': 63}
0.50-0.65 (Uncertain)     | 14         | 57.1%           | {'Hierarchical': 14}
0.65-0.80 (Moderate)      | 20         | 30.0%           | {'Truncated': 20}
0.80-0.95 (High)          | 2          | 0.0%            | {'Truncated': 2}

🔍 TABLE 2: HIGH-CONFIDENCE FAILURES (The 'Why did it fail?' Autopsy)
Articles where the model was >75% confident, but the Human disagreed.
----------------------------------------------------------------------
Found 8 high-confidence disagreements:

❌ Machine: NEUTRAL (0.77 via Truncated) | 👤 Human: SUPPORTIVE
📰 Text Snippet: To rai